# makemore part 3 — Activations, Gradients, BatchNorm

跟敲用。这一集不加新的模型能力，全部是**让已有的 MLP 训得动、训得稳**。

数据和划分直接用共享包，别自己 shuffle：

```python
from nnzh.data import VOCAB_SIZE, bpc, build_dataset, build_vocab, load_words, split_words
```


## 0. 起点：把第三集的 MLP 搬过来

`block_size=3, n_embd=10, n_hidden=200`，val 3.0648 bpc。这一集的所有改动都拿它当基线。


## 1. 修初始 loss

现象：初始 loss 远高于 `ln 27 = 3.2958`。原因是 `W2`/`b2` 太大，logits 跨度过宽，
softmax 输出接近 one-hot，模型在**自信地犯错**。前几百步全浪费在把 logits 压回 0。


## 2. 修饱和的 tanh

看 `h` 的直方图：如果大量值挤在 ±1，tanh 的梯度 `1 - h**2` 就接近 0，
那些神经元**反向传播时是死的**。画 `(h.abs() > 0.99)` 的白图看有多少列全白。


## 3. Kaiming 初始化

把 `w1_scale=0.2` 这种手调数字换成有原理的 `gain / sqrt(fan_in)`。
tanh 的 gain 是 `5/3`。目标：每一层的输出方差和输入方差保持一致。


## 4. BatchNorm

不再依赖初始化把分布调对，而是**每个 batch 强行标准化**再用 `bngain`/`bnbias` 学回来。
注意两件事：训练/推理行为不同（running mean/std）；batch 内样本之间产生了耦合。


## 5. PyTorch 化：Linear / BatchNorm1d / Tanh 类

把散着的张量收进类里，为第五集的 `Sequential` 铺路。


## 6. 诊断工具（这一集真正的收获）

三张图，训练前必看：

1. **前向**：每层激活值的分布，看饱和比例
2. **反向**：每层梯度的分布，看有没有消失/爆炸
3. **update:data ratio**：`(lr*grad).std() / param.std()`，经验值应在 `1e-3` 附近

第 3 张最实用——它直接告诉你学习率该调大还是调小，比扫描省事。


## 7. 对比

把 val bpc 填进 `experiments/bpc.md`。这一集的收益可能很小（模型容量没变），
**重点是训练过程变健康了**，而不是数字变好看。诚实记录。
